In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ["OMP_NUM_THREADS"] = '1'
from mpi4py import MPI
import numpy as np
import quimb.tensor as qtn
import pickle
from functools import partial
import torch
import json
import autoray as ar
# ==============================================================================
from vmc_torch.experiment.vmap.vmap_utils import random_initial_config, sample_next_reuse, sample_next
from vmc_torch.experiment.vmap.vmap_models import (
    Transformer_fPEPS_Model_Cluster_reuse,
    Transformer_fPEPS_Model_Cluster,
)
from vmc_torch.experiment.vmap.vmap_modules import distributed_minres_solver, run_sampling_phase_reuse
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.experiment.tn_model import init_weights_to_zero
from vmc_torch.experiment.vmap.vmap_torch_utils import robust_svd_err_catcher_wrapper
from vmc_torch.optimizer import DecayScheduler
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")
# ==============================================================================
SVD_JITTER = 1e-16
driver = None
ar.register_function('torch','linalg.svd', lambda x: robust_svd_err_catcher_wrapper(x, jitter=SVD_JITTER, driver=driver))
# ==============================================================================
COMM = MPI.COMM_WORLD
RANK = COMM.Get_rank()
SIZE = COMM.Get_size()
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
torch.set_default_device("cpu")
torch.random.manual_seed(42 + RANK)
# ==============================================================================
# 1. Initialization & Configuration
# ==============================================================================
Lx, Ly = 8, 8
N_f = Lx * Ly - 8
D, chi = 8, 32
t, U = 1.0, 8.0

# Load PEPS
u1z2 = True
appendix = '_U1SU' if u1z2 else ''
params_path = f'{pwd}/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/Z2/D={D}/'
params = pickle.load(open(params_path + f'peps_su_params{appendix}.pkl', 'rb'))
skeleton = pickle.load(open(params_path + f'peps_skeleton{appendix}.pkl', 'rb'))
peps = qtn.unpack(params, skeleton)
for ts in peps.tensors: 
    ts.modify(data=ts.data.to_flat()*4)
for site in peps.sites: 
    peps[site].data._label = site
    peps[site].data.indices[-1]._linearmap = ((0, 0), (1, 0), (1, 1), (0, 1)) # Important for U1->Z2 fPEPS

# ==============================================================================
# Model Configuration (Define this FIRST)
# ==============================================================================
# Put all hyperparameters for initialization here
# Note: ftn (peps) is usually too large or an object, not suitable for json storage, only record the parameters used to generate peps (Lx, Ly, etc.)
model_config = {
    'max_bond': chi,
    'embed_dim': 16,
    'attn_depth': 1,
    'attn_heads': 4,
    'nn_hidden_dim': D, #peps.nsites,
    'init_perturbation_scale': 1e-3,
    'nn_eta': 1,
    'dtype_str': 'float64',
    'uniform_kernel': 0,
}
dtype_map = {'float64': torch.float64, 'float32': torch.float32}
model_dtype = dtype_map[model_config['dtype_str']]
init_kwargs = model_config.copy()
init_kwargs.pop('dtype_str')
# Model
# fpeps_model = Transformer_fPEPS_Model_Conv2d(
#     tn=peps,
#     dtype=model_dtype,
#     **init_kwargs
# )
fpeps_model = Transformer_fPEPS_Model_Cluster_reuse(
    tn=peps,
    dtype=model_dtype,
    contract_boundary_opts={
        'mode': 'mps',
        # 'equalize_norms': 1.0,
        'canonize': True,
    },
    **init_kwargs
)
fpeps_model1 = Transformer_fPEPS_Model_Cluster(
    tn=peps,
    dtype=model_dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
    },
    **init_kwargs
)
n_params = sum(p.numel() for p in fpeps_model.parameters())
if RANK == 0: 
    print(f'Model Params: {n_params}')

# Hamiltonian
H = spinful_Fermi_Hubbard_square_lattice_torch(
    Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=(N_f//2, N_f//2), no_u1_symmetry=False,
)

# VMC Hyperparams
Ns = int(6e3) 
B = 5
B_grad = B//2
vmc_steps = 50
init_step = 0
burn_in_steps = 5
learning_rate = 0.1
diag_shift = 1e-4
save_state_every = 10
scheduler = DecayScheduler(init_lr=learning_rate, decay_rate=0.9, patience=50, min_lr=1e-2)

# # Load Checkpoint
# file_path = f'{params_path}/{fpeps_model._get_name()}/chi={chi}/'
# fpeps_model.debug_file = file_path
# if init_step > 0:
#     ckpt_path = file_path + f'checkpoint_step_{init_step}.pt'
#     fpeps_model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
#     fpeps_model1.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
#     if RANK == 0: 
#         print(f'Loaded step {init_step}')
fpeps_model_params = fpeps_model.state_dict()
fpeps_model1.load_state_dict(fpeps_model_params)

fxs0 = torch.stack([random_initial_config(N_f, peps.nsites) for _ in range(B)]).to(torch.long)
fpeps_model.cache_bMPS_skeleton(fxs0[0])

/tmp/ipykernel_227165/2957179319.py:15: FutureWarning: The vmap_models module is deprecated and will be removed in future versions. Please use the updated models in .models directory instead.
  from vmc_torch.experiment.vmap.vmap_models import (


 -> [Init] LocalClusterBackflow: Clamping output weights to scale 0.001
 -> [Init] LocalClusterBackflow: Clamping output weights to scale 0.001
Model Params: 3336192


In [2]:
import time
fxs = fxs0.clone()
with torch.no_grad():
    t0 = time.time()
    # reuse_amps = fpeps_model(fxs)
    reuse_fxs, reuse_amps = sample_next_reuse(fxs0.clone(), fpeps_model, H.graph, show_pbar=True)
    t1 = time.time()
    # amps1 = fpeps_model1(fxs)
    fxs1, amps1 = sample_next(fxs0.clone(), fpeps_model1, H.graph, show_pbar=True)
    t2 = time.time()

reuse_amps, t1-t0, amps1, t2-t1

MH Sampling: 100%|██████████| 112/112 [02:29<00:00,  1.33s/it]


(tensor([-4.1154e-04, -1.5461e-01,  6.6820e+02, -8.7184e-05, -1.2336e+02],
        dtype=torch.float64),
 70.209477186203,
 tensor([-2.2290e-02, -9.6379e-02, -1.2176e+04,  4.7414e-05, -2.1278e-01],
        dtype=torch.float64),
 150.95865631103516)

In [3]:
with torch.no_grad():
    reuse_amps1 = fpeps_model(reuse_fxs)
    amps11 = fpeps_model1(fxs1)
reuse_amps1, amps11

(tensor([-4.1128e-04, -1.5459e-01,  6.6821e+02, -8.7321e-05, -1.2336e+02],
        dtype=torch.float64),
 tensor([-2.2290e-02, -9.6379e-02, -1.2176e+04,  4.7414e-05, -2.1278e-01],
        dtype=torch.float64))

In [4]:
torch.allclose(fxs1, fxs)

False